# 01 · Extracción de ofertas públicas: Talento Perú

**Etapa 1 del pipeline: ingestión** · Responsable: **Integrante 1** (organización del proyecto y pipeline de datos)

| | |
|---|---|
| **Lee** | Portal Talento Perú (SERVIR), sección "Ofertas Laborales (Convocatoria Pública)", solo si `EJECUTAR_EXTRACCION = True` |
| **Escribe** | Un archivo por corte: `data/raw/talento_peru_ofertas_AAAAMMDD.csv` y su estado `talento_peru_estado_AAAAMMDD.json` |
| **Dataset inicial** | `data/raw/talento_peru_ofertas.csv`: corte del 17 y 18 de septiembre de 2026, 1010 ofertas y 12 columnas (congelado) |
| **Valida** | Contrato `raw` de `src/validacion.py` |

Este notebook documenta cómo se obtuvo el dataset inicial y permite generar un corte nuevo. El código vive en `src/extraccion.py`, de modo que el notebook, el orquestador `run_pipeline.py` y las pruebas automáticas usan exactamente las mismas funciones.

## Cambios respecto a la Parte 1

| # | Problema en la Parte 1 | Corrección |
|---|---|---|
| 1 | Volver a ejecutar agregaba filas al mismo CSV y mezclaba fechas de corte | Un archivo por corte; el extractor se niega a escribir sobre el dataset inicial |
| 2 | `fecha_extraccion_utc` se calculaba una sola vez, pero el CSV entregado tiene una marca por página (105 valores) | Se registra por página, igual que en la corrida que generó los datos |
| 3 | Solo funcionaba con Chromium + ChromeDriver en el PATH (Linux o Colab) | En Windows y macOS usa Google Chrome con Selenium Manager, que descarga el driver |
| 4 | El archivo de estado se definía pero nunca se escribía | Se guarda al terminar, o si la corrida falla; el del corte inicial se reconstruye desde el CSV |
| 5 | Rutas relativas a `..` que fallaban según desde dónde se abriera el notebook | Todas las rutas salen de `src/config.py` |
| 6 | El botón "Siguiente" dependía de un id autogenerado por JSF | Si el id cambia, el botón se busca por su texto |

Las correcciones están cubiertas por las pruebas de `tests/` (parseo, hash, validación y navegación con Selenium sobre un portal simulado): `python -m pytest`.

## Preparación

Se ubica la raíz del repositorio para importar el código compartido de `src/`.

In [1]:
import sys
from pathlib import Path

# Ubica la raíz del repositorio (la carpeta que contiene src/config.py),
# así el notebook funciona desde notebooks/, desde la raíz o en Colab.
RAIZ = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "config.py").exists()), None)
if RAIZ is None:
    raise FileNotFoundError(
        "No se encontró la raíz del repositorio. Abra el notebook desde la carpeta del proyecto; "
        "en Colab, ejecute antes: !git clone <url del repositorio> y %cd <carpeta del repositorio>"
    )
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import pandas as pd
from src import config, validacion
import inspect
import json

from src import extraccion

## Parámetros

La extracción está **desactivada por defecto**. Las ofertas del dataset inicial terminaban de publicarse el 30 de septiembre y el 1 de octubre de 2026, así que una nueva corrida no reproduce ese corte: genera otro distinto. Por eso el dataset inicial se conserva tal cual y un corte nuevo se guarda en un archivo aparte con su fecha.

In [2]:
# Poner en True solo para generar un corte NUEVO (requiere Chrome y conexión a internet).
EJECUTAR_EXTRACCION = False
LIMITE = 1010         # ofertas únicas a reunir en el corte
PAUSA_SEGUNDOS = 2    # pausa entre páginas para no sobrecargar el portal público
ARCHIVO_CORTE = None  # None = data/raw/talento_peru_ofertas_<fecha de hoy>.csv

## Cómo se convierte una tarjeta en un registro

Cada oferta del listado es una tarjeta `.cuadro-vacantes`. El puesto está en `.titulo-vacante label`, la entidad en `.nombre-entidad .detalle-sp` y el resto de campos en pares `span.sub-titulo` / `span.detalle-sp`.

Durante el parseo se aplica una limpieza mínima: se normalizan espacios (incluido `\xa0`), las vacantes pasan a entero, la remuneración a número decimal (`"S/. 5,670.00"` → `5670.0`) y las fechas de `DD/MM/AAAA` a `AAAA-MM-DD`. Si falta alguno de los 8 campos principales, la tarjeta se descarta.

El identificador `id_oferta_hash` es el SHA-256 de los 8 campos principales unidos por `|`, recortado a 24 caracteres. Al ser determinista, permite reanudar una corrida sin duplicar y verificar que el dataset no fue alterado.

In [3]:
print(inspect.getsource(extraccion.extraer_oferta))

def extraer_oferta(tarjeta, pagina: int, fecha_extraccion: str) -> dict | None:
    """Convierte una tarjeta HTML en un registro. Devuelve None si falta un campo principal."""
    titulo = tarjeta.select_one(".titulo-vacante label")
    entidad = tarjeta.select_one(".nombre-entidad .detalle-sp")

    oferta = {
        "puesto": limpiar_texto(titulo.get_text(" ", strip=True) if titulo else ""),
        "entidad": limpiar_texto(entidad.get_text(" ", strip=True) if entidad else ""),
        "ubicacion": obtener_valor(tarjeta, "Ubicación:"),
        "numero_convocatoria": obtener_valor(tarjeta, "Número de Convocatoria:"),
        "vacantes": convertir_entero(obtener_valor(tarjeta, "Cantidad de Vacantes:")),
        "remuneracion_pen": convertir_numero(obtener_valor(tarjeta, "Remuneración:")),
        "fecha_inicio_publicacion": convertir_fecha(obtener_valor(tarjeta, "Fecha Inicio de Publicación:")),
        "fecha_fin_publicacion": convertir_fecha(obtener_valor(tarjeta, "Fecha Fin de Publ

## Cómo se recorre el listado

`extraccion.extraer_listado()` sigue estos pasos:

1. Abre el portal con Selenium en modo sin interfaz y espera hasta 20 segundos a que aparezcan las tarjetas (el portal usa JSF/PrimeFaces y carga el contenido con AJAX).
2. En cada página lee el indicador "Página X de Y", registra la hora UTC y procesa las tarjetas con BeautifulSoup.
3. Omite las ofertas cuyo hash ya está en el archivo del corte y agrega las nuevas al CSV en el momento (almacenamiento incremental).
4. Se detiene al llegar al límite, a la última página o si el botón "Siguiente" está desactivado.
5. Espera 2 segundos, pulsa "Siguiente" y espera a que cambie el indicador de página.
6. Al terminar, o si ocurre un error, guarda el estado de la corrida en JSON.

## Ejecución

In [4]:
if EJECUTAR_EXTRACCION:
    archivo = Path(ARCHIVO_CORTE) if ARCHIVO_CORTE else extraccion.ruta_corte()
    estado = extraccion.extraer_listado(LIMITE, PAUSA_SEGUNDOS, archivo)
    print(json.dumps(estado, ensure_ascii=False, indent=2))
    validacion.validar("raw", archivo).exigir()
else:
    print("Extracción desactivada: se trabaja con el dataset inicial congelado en data/raw/.")

Extracción desactivada: se trabaja con el dataset inicial congelado en data/raw/.


## Estado del corte inicial

La corrida de la Parte 1 no escribió su archivo de estado. Se reconstruye a partir de las columnas de trazabilidad del propio CSV (`pagina_origen` y `fecha_extraccion_utc`) y se marca como reconstruido.

In [5]:
if config.ARCHIVO_ESTADO_RAW.exists():
    estado_inicial = json.loads(config.ARCHIVO_ESTADO_RAW.read_text(encoding=config.ENCODING))
else:
    estado_inicial = {
        "version_extractor": "1.0 (Parte 1)",
        "reconstruido": True,
        "nota": "La corrida de la Parte 1 no escribió su archivo de estado; estos datos se derivan del CSV.",
        "fuente_url": config.URL_FUENTE,
        "limite": 1010,
        "pausa_segundos": 2,
        **extraccion.resumen_corte(config.ARCHIVO_RAW),
    }
    extraccion.guardar_estado(estado_inicial, config.ARCHIVO_ESTADO_RAW)

print(json.dumps(estado_inicial, ensure_ascii=False, indent=2))

{
  "version_extractor": "1.0 (Parte 1)",
  "reconstruido": true,
  "nota": "La corrida de la Parte 1 no escribió su archivo de estado; estos datos se derivan del CSV.",
  "fuente_url": "https://app.servir.gob.pe/DifusionOfertasExterno/faces/consultas/ofertas_laborales.xhtml",
  "limite": 1010,
  "pausa_segundos": 2,
  "archivo": "talento_peru_ofertas.csv",
  "ofertas": 1010,
  "ofertas_unicas": 1010,
  "paginas_con_ofertas": 105,
  "pagina_inicial": 1,
  "pagina_final": 105,
  "primera_pagina_utc": "2026-09-18T05:58:55.931516+00:00",
  "ultima_pagina_utc": "2026-09-18T06:05:39.198174+00:00",
  "segundos_entre_primera_y_ultima_pagina": 403.3
}


## Validación del dataset inicial

El contrato `raw` exige las 12 columnas en orden, ningún nulo, `id_oferta_hash` único, tipos correctos y que el hash de cada fila se pueda recalcular desde sus campos principales (prueba de linaje con la extracción). Los valores sospechosos se informan como advertencias para que la etapa de limpieza los trate y justifique.

In [6]:
resultado_raw = validacion.validar("raw").exigir()

[OK] raw: data/raw/talento_peru_ofertas.csv (1010 filas, 12 columnas)
  ADVERTENCIA: 4 remuneraciones menores a S/ 1000 para revisar: [1.0, 1.0, 10.0, 200.0] (umbral aún no definido en config)
  ADVERTENCIA: 1 ofertas cuyo puesto es solo un número (p. ej. '1')


## Perfil rápido para las siguientes etapas

No es el análisis exploratorio (eso corresponde al notebook 05): es la información que las etapas 2 a 4 necesitan para empezar.

In [7]:
df = validacion.cargar(config.ARCHIVO_RAW)
print(f"{len(df)} ofertas, {df.shape[1]} columnas, {df['vacantes'].sum()} vacantes en total")
df.info()

1010 ofertas, 12 columnas, 1762 vacantes en total
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1010 entries, 0 to 1009
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   id_oferta_hash            1010 non-null   object 
 1   puesto                    1010 non-null   object 
 2   entidad                   1010 non-null   object 
 3   ubicacion                 1010 non-null   object 
 4   numero_convocatoria       1010 non-null   object 
 5   vacantes                  1010 non-null   int64  
 6   remuneracion_pen          1010 non-null   float64
 7   fecha_inicio_publicacion  1010 non-null   object 
 8   fecha_fin_publicacion     1010 non-null   object 
 9   pagina_origen             1010 non-null   int64  
 10  fuente_url                1010 non-null   object 
 11  fecha_extraccion_utc      1010 non-null   object 
dtypes: float64(1), int64(2), object(9)
memory usage: 94.8+ KB


In [8]:
df[["vacantes", "remuneracion_pen"]].describe()

,vacantes,remuneracion_pen
count,1010.000000,1010.000000
mean,1.744554,5130.148980
std,3.206796,2503.316623
min,1.000000,1.000000
25%,1.000000,3100.000000
50%,1.000000,5353.295000
75%,1.000000,6500.000000
max,72.000000,21000.000000


In [9]:
departamento = df["ubicacion"].apply(lambda v: v.split(" - ")[0])
BM_1057 = df["numero_convocatoria"].str.contains("D.LEG 1057")

resumen = pd.Series({
    "departamentos distintos": len(departamento.unique()),
    "entidades distintas": len(df["entidad"].unique()),
    "puestos distintos": len(df["puesto"].unique()),
    "ofertas cuya convocatoria contiene 'D.LEG 1057'": BM_1057.sum(),
    "fechas de inicio de publicación distintas": len(df["fecha_inicio_publicacion"].unique()),
})
resumen

departamentos distintos                             23
entidades distintas                                114
puestos distintos                                  842
ofertas cuya convocatoria contiene 'D.LEG 1057'    535
fechas de inicio de publicación distintas            2
dtype: int64

In [10]:
departamento.value_counts().head(5)

ubicacion
LIMA           439
LA LIBERTAD    115
CUSCO           97
HUANUCO         60
JUNIN           34
Name: count, dtype: int64

## Resultado de la etapa

- **Dataset inicial validado:** `data/raw/talento_peru_ofertas.csv`, 1010 ofertas únicas, 12 columnas, sin nulos y con hash reproducible en todas las filas.
- **Para la limpieza (Integrante 2):** remuneraciones no plausibles y un puesto con valor `"1"`, señalados en las advertencias de la validación.
- **Para la transformación (Integrante 3):** `ubicacion` tiene el formato "DEPARTAMENTO - DISTRITO" y se puede separar.
- **Para Feature Engineering (Integrante 4):** `numero_convocatoria` contiene el régimen laboral; solo hay dos fechas de inicio de publicación, así que no se puede estudiar una tendencia temporal.

**Siguiente etapa:** `02_limpieza_datos.ipynb` lee `data/raw/talento_peru_ofertas.csv`.